# Step 1 — All-Section IFRS Requirement Mapping

This notebook maps the five BANK01 section payloads to their IFRS S1/S2 requirement files.

It uses the same project paths as the main IFRS report-generation notebook:

```text
<project>/notebooks/gen_data/payloads_risk/
<project>/notebooks/gen_data/IFRS/ifrs_requirements_kb_outputs_final/section_by_section_requirements/json/
<project>/notebooks/gen_data/generated_reports/agentic_ifrs_report/00_requirement_mapping/
```

The notebook can be launched either:

- from the project root containing the `notebooks` folder; or
- directly from the `notebooks` folder.

Environment variables remain available as optional overrides:

- `PAYLOAD_DIR`
- `IFRS_REQUIREMENTS_DIR`
- `GENERATION_OUTPUT_DIR`
- `MAPPING_OUTPUT_DIR`

The notebook performs **mapping only**. It does not run report writing, claims extraction, judges, revisions, or approval logic.

## Strict status meanings

- `covered`: every required evidence slot resolves to non-empty payload paths.
- `partially_covered`: some evidence exists, but an explicit process, policy, disaggregation, or other required slot is absent.
- `not_available_in_payload`: no required evidence slot is supported.
- `handled_by_report_design`: a presentation, duplication, cross-reference, or report-structure control.
- `conditional_not_triggered`: the requirement applies only if a specified event occurs, and that event is not represented in the payload.
- `not_applicable_to_entity_scope`: the payload does not contain the business activity to which the requirement applies.

The mapper uses requirement-specific deterministic contracts. It does not use an LLM or unrestricted lexical similarity to select evidence paths.


In [ ]:
import json
import math
import os
import re
from collections import Counter
from pathlib import Path


# ============================================================
# PROJECT PATHS — SAME STRUCTURE AS THE MAIN IFRS NOTEBOOK
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / "notebooks").exists():
    NOTEBOOK_DIR = (CURRENT_DIR / "notebooks").resolve()
else:
    NOTEBOOK_DIR = CURRENT_DIR

GEN_DATA_DIR = NOTEBOOK_DIR / "gen_data"

PAYLOAD_DIR = Path(
    os.getenv(
        "PAYLOAD_DIR",
        str(GEN_DATA_DIR / "payloads_risk"),
    )
).resolve()

REQUIREMENTS_DIR = Path(
    os.getenv(
        "IFRS_REQUIREMENTS_DIR",
        str(
            GEN_DATA_DIR
            / "IFRS"
            / "ifrs_requirements_kb_outputs_final"
            / "section_by_section_requirements"
            / "json"
        ),
    )
).resolve()

GENERATION_OUTPUT_DIR = Path(
    os.getenv(
        "GENERATION_OUTPUT_DIR",
        str(GEN_DATA_DIR / "generated_reports" / "agentic_ifrs_report"),
    )
).resolve()

MAPPING_OUTPUT_DIR = Path(
    os.getenv(
        "MAPPING_OUTPUT_DIR",
        str(GENERATION_OUTPUT_DIR / "00_requirement_mapping"),
    )
).resolve()


# Exact files used by this mapping stage.
RESOLVED_FILES = {
    "general_requirements": {
        "payload": PAYLOAD_DIR / "payload_BANK01_general_requirements.json",
        "requirements": REQUIREMENTS_DIR / "general_requirements_requirements.json",
    },
    "governance": {
        "payload": PAYLOAD_DIR / "payload_BANK01_governance.json",
        "requirements": REQUIREMENTS_DIR / "governance_requirements.json",
    },
    "strategy": {
        "payload": PAYLOAD_DIR / "payload_BANK01_strategy.json",
        "requirements": REQUIREMENTS_DIR / "strategy_requirements.json",
    },
    "risk_management": {
        "payload": PAYLOAD_DIR / "payload_BANK01_risk_management.json",
        "requirements": REQUIREMENTS_DIR / "risk_management_requirements.json",
    },
    "metrics_and_targets": {
        "payload": PAYLOAD_DIR / "payload_BANK01_metrics_targets.json",
        "requirements": REQUIREMENTS_DIR / "metrics_and_targets_requirements.json",
    },
}


def validate_input_paths():
    missing = []

    if not PAYLOAD_DIR.exists():
        missing.append(f"Payload directory does not exist: {PAYLOAD_DIR}")

    if not REQUIREMENTS_DIR.exists():
        missing.append(f"Requirements directory does not exist: {REQUIREMENTS_DIR}")

    for section, files in RESOLVED_FILES.items():
        for kind, path in files.items():
            if not path.exists():
                missing.append(f"{section} {kind} file not found: {path}")

    if missing:
        raise FileNotFoundError(
            "The mapping notebook could not find the required project inputs.\n\n"
            + "\n".join(f"- {item}" for item in missing)
            + "\n\nCurrent working directory: "
            + str(CURRENT_DIR)
            + "\nResolved notebook directory: "
            + str(NOTEBOOK_DIR)
            + "\n\nRun the notebook from the project root or the notebooks folder, "
              "or override PAYLOAD_DIR / IFRS_REQUIREMENTS_DIR."
        )


print("Current working directory:", CURRENT_DIR)
print("Notebook directory:", NOTEBOOK_DIR)
print("Payload directory:", PAYLOAD_DIR)
print("Requirements directory:", REQUIREMENTS_DIR)
print("Generation output directory:", GENERATION_OUTPUT_DIR)
print("Mapping output directory:", MAPPING_OUTPUT_DIR)

for section, files in RESOLVED_FILES.items():
    print(f"\n{section}")
    print("  payload:", files["payload"])
    print("  requirements:", files["requirements"])

validate_input_paths()
print("\nINPUT PATH VALIDATION: PASS")


In [ ]:
GENERIC_LEAVES = {"bank_id", "reporting_year", "summary_id", "id", "country", "is_synthetic", "ifrs_s2_para_evidence"}

def load_json(path):
    with open(path,encoding='utf-8') as f: return json.load(f)


def iter_requirements(doc):
    out=[]
    for std,block in doc["standards"].items():
        out.extend(block["requirements"])
    return out


def is_empty(v):
    if v is None: return True
    if isinstance(v,float) and math.isnan(v): return True
    if isinstance(v,str) and v.strip().lower() in {"","none","null","nan","n/a","na","not_applicable"}: return True
    if isinstance(v,(list,dict)) and not v: return True
    return False


def flatten_records(obj, prefix="", context=None):
    context=dict(context or {})
    rows=[]
    if isinstance(obj,dict):
        local=dict(context)
        # record-level context
        for k in ("reporting_year","risk_id","target_id","scenario_id","opportunity_id","effect_id","investment_id","scope3_id","method_id","cons_id","meeting_id","tp_id","resilience_id","node_id","physical_risk_id"):
            if k in obj:
                local[k]=obj[k]
        for k,v in obj.items():
            p=f"{prefix}.{k}" if prefix else k
            if isinstance(v,(dict,list)):
                rows.extend(flatten_records(v,p,local))
            else:
                rows.append({"path":p,"value":v,"leaf":k,"root":p.split(".",1)[0].split("[",1)[0],"context":local})
    elif isinstance(obj,list):
        for i,v in enumerate(obj):
            p=f"{prefix}[{i}]"
            rows.extend(flatten_records(v,p,context))
    return rows


def txt(req):
    return " ".join([
        str(req.get("requirement_text","")),
        str(req.get("official_section_heading","")),
    ]).lower().replace("’","'").replace("–","-").replace("—","-")


def has(t, *phrases): return any(p.lower() in t for p in phrases)


def slot(name, roots, path_terms=(), value_terms=(), description="", min_paths=1, max_paths=5, allow_generic=False):
    return {
        "name":name, "roots":set(roots), "path_terms":tuple(x.lower() for x in path_terms),
        "value_terms":tuple(x.lower() for x in value_terms), "description":description or name,
        "min_paths":min_paths, "max_paths":max_paths, "allow_generic":allow_generic,
    }


def choose_paths(flat, sl, reporting_year, comparative=False):
    candidates=[]
    for e in flat:
        if e["root"] not in sl["roots"]: continue
        if is_empty(e["value"]): continue
        if e["leaf"] in GENERIC_LEAVES and not sl["allow_generic"]: continue
        p=e["path"].lower().replace("_"," ")
        pv=p+" "+(str(e["value"]).lower().replace("_"," ") if isinstance(e["value"],(str,int,float,bool)) else "")
        path_hit=any(term.replace("_"," ") in p for term in sl["path_terms"]) if sl["path_terms"] else False
        value_hit=any(term.replace("_"," ") in pv for term in sl["value_terms"]) if sl["value_terms"] else False
        if not (path_hit or value_hit): continue
        rec_year=e["context"].get("reporting_year")
        year_score=0
        if comparative:
            if rec_year in ([reporting_year]+[reporting_year-1,reporting_year-2]): year_score=2
        else:
            if rec_year == reporting_year: year_score=3
            elif rec_year is None: year_score=1
            else: year_score=-1
        specificity=sum(1 for term in sl["path_terms"] if term.replace("_"," ") in p)
        if value_hit: specificity += 1
        candidates.append((year_score,specificity,len(p),e))
    candidates.sort(key=lambda x:(x[0],x[1],-x[2]), reverse=True)
    out=[]; seen=set()
    for _,_,_,e in candidates:
        # permit multiple records but avoid duplicate same leaf/value recordlessly
        key=(e["path"])
        if key in seen: continue
        seen.add(key); out.append(e["path"])
        if len(out)>=sl["max_paths"]: break
    return out


In [ ]:
def contract_general(req):
    t=txt(req); rid=req["requirement_id"]
    if has(t,"avoid duplication","clear language","clearly structured sentences","identify its sustainability-related financial disclosures clearly",
           "shall not obscure material information","present such information as clearly as possible","presented as a coherent whole",
           "included by cross-reference","cross-reference shall","precisely specified part","same time as its related financial statements",
           "location of disclosures","part of its general purpose financial reports"):
        return {"treatment":"handled_by_report_design","name":"report_presentation_and_cross_reference","slots":[]}
    if has(t,"explicit and unreserved statement of compliance"):
        return {"treatment":"handled_by_report_design","name":"compliance_statement_control","slots":[]}
    if has(t,"changes the end of its reporting period","after the end of the reporting period","interim sustainability-related"):
        return {"treatment":"conditional_not_triggered","name":"reporting_period_event_condition","slots":[]}
    if has(t,"law or regulation prohibits","commercially sensitive","prohibited from using the exemption","law or regulation might specify"):
        return {"treatment":"conditional_not_triggered","name":"legal_disclosure_condition","slots":[]}
    if has(t,"material prior period error","identifies a material error","correct material prior period errors","restate the comparative"):
        return {"treatment":"conditional_not_triggered","name":"prior_period_error_condition","slots":[]}
    if has(t,"same reporting entity as the related financial statements"):
        return {"name":"reporting_entity_boundary","slots":[
            slot("reporting_entity",["general_requirements_context","bank"],["reporting_entity","bank_name"],description="Reporting entity identity"),
            slot("boundary",["general_requirements_context","bank"],["boundary_type"],description="Reporting boundary"),
        ]}
    if has(t,"presentation currency","currency is specified as the unit of measure"):
        return {"name":"presentation_currency","slots":[slot("currency",["general_requirements_context","bank"],["reporting_currency"],description="Presentation currency")]}
    if has(t,"reporting period","preceding period","comparative information"):
        return {"name":"reporting_period_and_comparatives","comparative":True,"slots":[
            slot("reporting_period",["general_requirements_context","bank","metadata"],["reporting_period_end","fiscal_year_end","reporting_year"],description="Reporting period",allow_generic=True),
            slot("comparatives",["general_requirements_context","metadata","financial_summary","scope1","scope2","financed_emissions"],["comparative_years","reporting_year","total_assets","scope1_total","scope2_location","financed_em"],description="Comparative period evidence",min_paths=2,max_paths=6,allow_generic=True)
        ]}
    if has(t,"measurement uncertainty","uncertainties affecting the amounts","assumptions, approximations and judgements","sources of measurement uncertainty","amounts that it has disclosed that are subject"):
        return {"name":"measurement_uncertainty","coverage_cap":"partially_covered","slots":[
            slot("uncertainty_sources",["metadata","general_requirements_context"],["data_gaps","reason","instruction","pcaf_methodology","scope2_rec_reconciliation","vehicles_correction"],description="Identified uncertainty sources",min_paths=2,max_paths=6),
            slot("affected_amounts",["scope1","scope2","scope3_travel","financed_emissions","reporting_kpis"],["scope1","scope2","scope3","financed_emissions","carbon_intensity"],description="Amounts affected by estimation or data limitations",min_paths=1,max_paths=5),
            slot("high_uncertainty_designation",["general_requirements_context","metadata"],["high_measurement_uncertainty","uncertainty_level","high_uncertainty"],description="Explicit identification of high-uncertainty amounts")
        ]}
    if has(t,"judgements, apart from those involving estimations","made in the process of preparing"):
        return {"name":"preparation_judgements","slots":[slot("judgement_basis",["metadata","general_requirements_context"],["pcaf_methodology","risk_rating_methodology","scope2_rec_reconciliation","vehicles_correction"],description="Methodological judgements",min_paths=2,max_paths=6)]}
    if has(t,"specific standards, pronouncements, industry practice","sources of guidance","sasb standards","industry(s) specified"):
        slots=[slot("standards_basis",["general_requirements_context","metadata","bank"],["standards_basis","regulatory_regime","pcaf_methodology"],description="Applied reporting and methodology sources",min_paths=1,max_paths=5)]
        if has(t,"sasb","industry(s) specified","industry practice"):
            slots.append(slot("industry_guidance_considered",["general_requirements_context","metadata"],["sasb","industry_guidance","industry_standard"],description="Industry guidance considered"))
        return {"name":"sources_of_guidance","coverage_cap":"partially_covered" if len(slots)>1 else None,"slots":slots}
    if has(t,"apply this standard","ifrs sustainability disclosure standards","scope"):
        return {"name":"standards_basis","slots":[slot("standards_basis",["general_requirements_context","bank"],["standards_basis","regulatory_regime"],description="Standards basis")]}
    if has(t,"fair presentation","complete, neutral and accurate","comparable, verifiable, timely and understandable",
           "neutral depiction","shall be accurate","enhances comparability","enhances its verifiability"):
        return {"name":"fair_presentation_controls","coverage_cap":"partially_covered","slots":[
            slot("assurance",["general_requirements_context"],["external_assurance","assurance_provider","assurance_scope","assurance_standard"],description="External assurance evidence",min_paths=2,max_paths=4),
            slot("data_controls",["general_requirements_context","metadata"],["source_systems","coherence_fixes_applied","vehicles_correction","scope2_rec_reconciliation"],description="Data lineage and coherence controls",min_paths=1,max_paths=5),
            slot("quality_policy",["general_requirements_context"],["fair_presentation_policy","quality_characteristics","neutrality_control","accuracy_control"],description="Explicit fair-presentation and quality policy")
        ]}
    if has(t,"material information","materiality","material in the context","reassess its materiality"):
        return {"name":"materiality_process","coverage_cap":"partially_covered","slots":[
            slot("materiality_method",["general_requirements_context","metadata"],["materiality","materiality_process","materiality_assessment"],description="Entity materiality process"),
            slot("material_outputs",["reporting_kpis","targets","financial_summary"],["high_carbon_sector_exposure","fossil_fuel_exposure","climate_capex","target_summary"],description="Material disclosure outputs")
        ]}
    if has(t,"connections between","connected information","consistent data, assumptions, and units","financial statements to which"):
        return {"name":"connected_information","coverage_cap":"partially_covered","slots":[
            slot("financial_connections",["financial_summary","reporting_kpis","general_requirements_context"],["total_assets","total_loans","climate_capex","climate_opex","reporting_currency"],description="Financial and sustainability data connections",min_paths=2,max_paths=6),
            slot("source_lineage",["general_requirements_context"],["source_systems"],description="Source-system lineage"),
            slot("explicit_consistency_process",["general_requirements_context"],["financial_statement_consistency","consistent_assumptions","reconciliation_process","cross_reference"],description="Explicit consistency and connection process")
        ]}
    if has(t,"governance-the governance processes","strategy-the approach","risk management-the processes","metrics and targets-the entity's performance"):
        return {"treatment":"handled_by_report_design","name":"core_content_section_structure","slots":[]}
    if has(t,"risks and opportunities that could reasonably be expected to affect","affect the entity's cash flows","affect the entity's prospects"):
        return {"name":"risks_opportunities_and_prospects","coverage_cap":"partially_covered","slots":[
            slot("financial_effect_indicators",["financial_summary","reporting_kpis","targets"],["climate_capex","climate_opex","financed_emissions","carbon_intensity","target"],description="Entity-specific sustainability-related financial indicators",min_paths=2,max_paths=6),
            slot("identified_risks_opportunities",["climate_risk_register","climate_opportunities"],["risk_name","opportunity_type"],description="Explicit identified sustainability risks and opportunities")
        ]}
    return {"treatment":"not_available_in_payload","name":"no_strict_general_requirements_contract","slots":[]}


_old_contract_general = contract_general

def contract_general(req):
    rid=req["requirement_id"]
    exact_design={
        "IFRS_S1_B32_C01":"material_information_despite_legal_permission",
        "IFRS_S1_B42_C02":"avoid_duplication_connected_information",
        "IFRS_S1_D26_C01":"avoid_boilerplate",
        "IFRS_S1_D29_C01":"avoid_obscuring_material_information",
        "IFRS_S1_D31_C01":"coherent_presentation",
    }
    if rid in exact_design:
        return {"treatment":"handled_by_report_design","name":exact_design[rid],"slots":[]}
    if rid in {"IFRS_S1_15_C02","IFRS_S1_D20_C01","IFRS_S1_D24_C01"}:
        return {"name":"fair_presentation_controls","coverage_cap":"partially_covered","slots":[
            slot("assurance",["general_requirements_context"],["external_assurance","assurance_provider","assurance_scope","assurance_standard"],description="External assurance evidence",min_paths=2,max_paths=4),
            slot("data_controls",["general_requirements_context","metadata"],["source_systems","coherence_fixes_applied","vehicles_correction","scope2_rec_reconciliation"],description="Data lineage and coherence controls",min_paths=1,max_paths=5),
            slot("quality_policy",["general_requirements_context"],["fair_presentation_policy","quality_characteristics","neutrality_control","accuracy_control"],description="Explicit fair-presentation and quality policy")
        ]}
    if rid in {"IFRS_S1_B20_C01","IFRS_S1_B22_C01","IFRS_S1_B22_C02"}:
        return {"name":"materiality_process","coverage_cap":"partially_covered","slots":[
            slot("materiality_method",["general_requirements_context","metadata"],["materiality","materiality_process","materiality_assessment"],description="Entity materiality process"),
            slot("material_outputs",["reporting_kpis","targets","financial_summary"],["high_carbon_sector_exposure","fossil_fuel_exposure","climate_capex","target_summary"],description="Material disclosure outputs")
        ]}
    if rid in {"IFRS_S1_D32_C01","IFRS_S1_D33_C01"}:
        return {"name":"connected_information","coverage_cap":"partially_covered","slots":[
            slot("financial_connections",["financial_summary","reporting_kpis","general_requirements_context"],["total_assets","total_loans","climate_capex","climate_opex","reporting_currency"],description="Financial and sustainability data connections",min_paths=2,max_paths=6),
            slot("source_lineage",["general_requirements_context"],["source_systems"],description="Source-system lineage"),
            slot("explicit_consistency_process",["general_requirements_context"],["financial_statement_consistency","consistent_assumptions","reconciliation_process","cross_reference"],description="Explicit consistency and connection process")
        ]}
    return _old_contract_general(req)


def contract_governance(req):
    t=txt(req); rid=req["requirement_id"]
    if rid=="IFRS_S2_7_C01":
        return {"treatment":"handled_by_report_design","name":"avoid_governance_duplication","slots":[]}
    if has(t,"terms of reference","mandates","role descriptions"):
        return {"name":"governance_mandate","slots":[
            slot("responsible_body",["governance","board_minutes"],["management_committee_name","committee_name","esg_committee_exists"],description="Responsible governance body",min_paths=2,max_paths=4),
            slot("documented_mandate",["governance","board_minutes"],["terms_of_reference","mandate","role_description","responsibilities","policy"],description="Documented governance mandate")
        ]}
    if has(t,"skills and competencies","appropriate skills"):
        return {"name":"governance_skills","slots":[
            slot("skills_evidence",["governance"],["board_climate_expertise_pct","skills_development_programme"],description="Skills and development evidence",min_paths=2,max_paths=3),
            slot("skills_assessment_process",["governance","board_minutes"],["skills_assessment","competency_assessment","training_plan"],description="Process for assessing skills adequacy")
        ]}
    if has(t,"how and how often","is informed"):
        return {"name":"governance_information_frequency","slots":[
            slot("reporting_frequency",["governance"],["climate_risk_reporting_to_board","board_full_meeting_frequency","esg_committee_meetings_per_year"],description="Board reporting frequency",min_paths=2,max_paths=4),
            slot("meeting_evidence",["board_minutes"],["meeting_date","climate_topics_discussed"],description="Climate agenda evidence",min_paths=2,max_paths=5)
        ]}
    if has(t,"major transactions","trade-offs","overseeing the entity's strategy"):
        return {"name":"strategy_transactions_tradeoffs","slots":[
            slot("strategy_and_transactions",["governance","board_minutes"],["major_transactions_climate_check","erm_integration_flag","transition_plan_review","scenario_analysis","physical_risk_update"],description="Strategy and major-transaction oversight",min_paths=2,max_paths=6),
            slot("tradeoffs",["board_minutes","governance"],["tradeoff","trade_off","trade-offs"],description="How trade-offs are considered")
        ]}
    if has(t,"oversees the setting of targets","monitors progress","remuneration policies"):
        return {"name":"targets_and_remuneration_oversight","slots":[
            slot("targets_progress",["board_minutes","reporting_kpis"],["net_zero_progress","target","target_summary"],description="Target setting and progress oversight",min_paths=1,max_paths=5),
            slot("remuneration",["governance","board_minutes"],["compensation","remuneration","exec_remuneration_esg_kpis"],description="Remuneration linkage",min_paths=2,max_paths=5)
        ]}
    if has(t,"delegated to a specific management-level position","management-level committee"):
        return {"name":"management_delegation","slots":[
            slot("management_body",["governance"],["management_committee_name","esg_committee_exists"],description="Management body or committee",min_paths=2,max_paths=3),
            slot("oversight_mechanism",["governance","board_minutes"],["oversight","reports_to_board","accountability"],description="How oversight is exercised")
        ]}
    if has(t,"management uses controls and procedures","integrated with other internal functions"):
        return {"name":"management_controls_integration","slots":[
            slot("controls",["governance"],["erm_integration_flag","major_transactions_climate_check"],description="Governance controls",min_paths=2,max_paths=3),
            slot("internal_function_integration",["governance","board_minutes"],["internal_functions","three_lines","control_framework","risk_function"],description="Integration with internal functions")
        ]}
    return {"treatment":"not_available_in_payload","name":"no_strict_governance_contract","slots":[]}


def _strategy_contract(name):
    catalog={
        "risks_opportunities":{"name":"identified_risks_and_opportunities","slots":[
            slot("risks",["climate_risk_register"],["risk_name","risk_description","risk_category"],description="Identified climate-related risks",min_paths=3,max_paths=8),
            slot("opportunities",["climate_opportunities"],["opportunity_type","category","description"],description="Identified climate-related opportunities",min_paths=2,max_paths=6)]},
        "risk_type":{"name":"risk_type_classification","slots":[slot("risk_types",["climate_risk_register"],["risk_name","risk_category"],value_terms=["physical","transition"],description="Physical/transition risk classification",min_paths=3,max_paths=8)]},
        "horizons":{"name":"risk_opportunity_horizons","slots":[slot("horizons",["climate_risk_register","climate_opportunities"],["time_horizon"],description="Risk and opportunity time horizons",min_paths=3,max_paths=8)]},
        "horizon_definitions":{"name":"horizon_definitions_and_planning","coverage_cap":"partially_covered","slots":[
            slot("horizon_values",["climate_risk_register","climate_opportunities","climate_scenarios","resilience_assessment"],["time_horizon","horizon","assessment_horizon"],description="Applied time horizons",min_paths=3,max_paths=8),
            slot("planning_link",["transition_plan","climate_scenarios"],["planning_horizon","strategic_planning","horizon_year"],description="Link to strategic planning horizons",min_paths=1,max_paths=4)]},
        "value_chain":{"name":"business_model_value_chain_effects","slots":[
            slot("value_chain",["value_chain_map"],["node_name","node_type","climate_risk_description","sustainability_theme"],description="Value-chain exposure and effects",min_paths=3,max_paths=8),
            slot("financial_exposure",["value_chain_map","reporting_kpis"],["financial_exposure","high_carbon_sector_exposure","fossil_fuel_exposure"],description="Financial exposure concentrations",min_paths=1,max_paths=5)]},
        "concentration":{"name":"value_chain_concentrations","slots":[
            slot("value_chain_locations",["value_chain_map"],["node_name","node_type","upstream_downstream","nace_link"],description="Value-chain locations",min_paths=3,max_paths=8),
            slot("concentrations",["value_chain_map","reporting_kpis"],["climate_exposure_type","materiality_flag","financial_exposure","high_carbon_sector_exposure","fossil_fuel_exposure"],description="Concentrations of risks and opportunities",min_paths=2,max_paths=6)]},
        "strategy_response":{"name":"strategy_response_and_transition_plan","slots":[
            slot("transition_plan",["transition_plan"],["has_transition_plan","net_zero_target_year","aligned_framework","key_assumptions","dependencies","resourcing"],description="Transition plan",min_paths=4,max_paths=8),
            slot("risk_responses",["climate_risk_register"],["mitigation_actions"],description="Risk responses",min_paths=3,max_paths=6),
            slot("targets",["targets"],["target_type","target_year","target_value","status"],description="Strategic targets",min_paths=2,max_paths=6)]},
        "progress":{"name":"prior_period_progress","slots":[slot("progress",["transition_plan","targets"],["prior_period_progress","actual_progress","target_progress","progress_basis"],description="Progress against previously disclosed plans",min_paths=1,max_paths=5)]},
        "tradeoffs":{"name":"strategy_tradeoffs","slots":[slot("tradeoffs",["transition_plan","climate_opportunities","climate_risk_register"],["tradeoff","trade_off","trade-offs"],description="Entity-specific trade-offs considered")]},
        "current_financial":{"name":"current_financial_effects","slots":[
            slot("current_effects",["climate_financial_effects"],["affected_statement","line_item","quantitative_effect_meur","qualitative_description"],description="Current financial effects",min_paths=4,max_paths=8),
            slot("current_financials",["financial_summary"],["climate_capex","climate_opex","total_revenue","net_profit"],description="Current-period financial indicators",min_paths=2,max_paths=5)]},
        "material_adjustment":{"name":"material_adjustment_next_period","slots":[
            slot("adjustment_flag",["climate_financial_effects"],["material_adjustment_next_period_flag"],description="Significant risk of material adjustment",min_paths=1,max_paths=5),
            slot("affected_items",["climate_financial_effects"],["affected_statement","line_item","linked_id"],description="Affected assets, liabilities or statement items",min_paths=2,max_paths=6)]},
        "anticipated_financial":{"name":"anticipated_financial_effects","slots":[
            slot("anticipated_effects",["climate_financial_effects"],["effect_timing","horizon","quantitative_effect_meur","qualitative_description"],description="Anticipated financial effects by horizon",min_paths=4,max_paths=8),
            slot("scenario_effects",["climate_scenarios"],["physical_risk_loss","transition_risk_loss","revenue_at_risk","stranded_assets"],description="Scenario-derived financial effects",min_paths=3,max_paths=8)]},
        "financial_planning":{"name":"financial_planning","slots":[
            slot("planned_resources",["transition_plan","financial_summary"],["resourcing_meur","resourcing_description","climate_capex","climate_opex"],description="Resources and financial planning",min_paths=2,max_paths=6),
            slot("planning_assumptions",["transition_plan","climate_scenarios"],["key_assumptions","dependencies","carbon_price_assumption"],description="Financial planning assumptions",min_paths=2,max_paths=6)]},
        "resilience":{"name":"climate_resilience","slots":[
            slot("resilience_results",["resilience_assessment","climate_scenarios"],["capacity_to_adjust","asset_redeployment_capacity","financial_resource_flexibility","resilience_assessment"],description="Resilience assessment results",min_paths=3,max_paths=8),
            slot("uncertainties",["resilience_assessment"],["significant_uncertainties"],description="Significant uncertainties",min_paths=1,max_paths=3)]},
        "scenario":{"name":"scenario_analysis","slots":[
            slot("scenario_design",["climate_scenarios"],["scenario_name","scenario_type","framework","temperature_outcome","horizon","horizon_year"],description="Scenario design and horizons",min_paths=4,max_paths=8),
            slot("methodology",["climate_scenarios"],["methodology_notes","scope_of_analysis","analysis_conducted_year"],description="Scenario methodology and scope",min_paths=2,max_paths=6)]},
        "inputs_capabilities":{"name":"scenario_inputs_capabilities","slots":[
            slot("inputs_assumptions",["climate_scenarios","transition_plan"],["carbon_price_assumption","gdp_growth_assumption","renewable_energy_share","key_assumptions","dependencies"],description="Scenario inputs and assumptions",min_paths=3,max_paths=8),
            slot("capabilities_resources",["transition_plan","resilience_assessment"],["resourcing","capacity_to_adjust","asset_redeployment_capacity","financial_resource_flexibility"],description="Capabilities and resources",min_paths=2,max_paths=6)]},
        "risk_process":{"name":"strategy_risk_identification_process","slots":[
            slot("identified_risks",["climate_risk_register"],["risk_name","risk_category","risk_description"],description="Identified risks",min_paths=3,max_paths=8),
            slot("risk_process_indicators",["climate_risk_register","climate_scenarios"],["likelihood_score","severity_score","risk_rating","scenario_analysis_link","monitoring_frequency"],description="Identification and assessment process evidence",min_paths=3,max_paths=8)]},
    }
    return catalog[name]


def contract_strategy(req):
    rid=req["requirement_id"]
    exact={}
    # summary clauses
    for x in ["IFRS_S1_29_C01","IFRS_S2_9_C01","IFRS_S1_30_C01","IFRS_S2_10_C01","IFRS_S2_11_C01","IFRS_S2_12_C01"]:
        exact[x]="risks_opportunities"
    exact["IFRS_S2_10_C02"]="risk_type"
    for x in ["IFRS_S1_30_C02","IFRS_S2_10_C03"]: exact[x]="horizons"
    for x in ["IFRS_S1_30_C03","IFRS_S2_10_C04"]: exact[x]="horizon_definitions"
    for x in ["IFRS_S1_29_C02","IFRS_S1_32_C01","IFRS_S2_9_C02","IFRS_S2_13_C01"]: exact[x]="value_chain"
    for x in ["IFRS_S1_32_C02","IFRS_S2_13_C02"]: exact[x]="concentration"
    for x in ["IFRS_S1_29_C03","IFRS_S1_33_C01","IFRS_S2_9_C03","IFRS_S2_14_C02","IFRS_S2_14_C03","IFRS_S2_14_C04","IFRS_S2_14_C05","IFRS_S2_14_C06","IFRS_S2_14_C07"]: exact[x]="strategy_response"
    for x in ["IFRS_S1_33_C02","IFRS_S2_14_C08"]: exact[x]="progress"
    exact["IFRS_S1_33_C03"]="tradeoffs"
    for x in ["IFRS_S1_34_C01","IFRS_S1_35_C01","IFRS_S2_15_C01","IFRS_S2_16_C01"]: exact[x]="current_financial"
    for x in ["IFRS_S1_35_C02","IFRS_S2_16_C02"]: exact[x]="material_adjustment"
    for x in ["IFRS_S1_34_C02","IFRS_S1_35_C06","IFRS_S2_15_C02","IFRS_S2_16_C06"]: exact[x]="anticipated_financial"
    for x in ["IFRS_S1_35_C04","IFRS_S1_35_C05","IFRS_S2_16_C04","IFRS_S2_16_C05"]: exact[x]="financial_planning"
    for x in ["IFRS_S1_29_C04","IFRS_S2_9_C04"]: 
        # umbrella current+anticipated
        exact[x]="anticipated_financial"
    for x in ["IFRS_S1_29_C05","IFRS_S1_42_C01","IFRS_S2_9_C05","IFRS_S2_B1_C01","IFRS_S2_B1_C02","IFRS_S2_B1_C03","IFRS_S2_B13_C01","IFRS_S2_B14_C01","IFRS_S2_B18_C01"]:
        exact[x]="resilience"
    for x in ["IFRS_S1_37_C01","IFRS_S1_37_C02","IFRS_S2_18_C01","IFRS_S2_18_C02","IFRS_S2_B6_C01","IFRS_S2_B11_C01","IFRS_S2_B12_C01"]:
        exact[x]="inputs_capabilities"
    for x in ["IFRS_S2_B2_C01","IFRS_S2_B2_C02","IFRS_S2_B3_C01","IFRS_S2_B4_C01","IFRS_S2_B8_C01","IFRS_S2_B8_C02","IFRS_S2_B17_C01"]:
        exact[x]="scenario"
    exact["IFRS_S2_B5_C01"]="risk_process"
    if rid=="IFRS_S2_23_C01":
        return {"treatment":"handled_by_report_design","name":"cross_section_metrics_reference","slots":[]}
    if rid in exact:
        return _strategy_contract(exact[rid])
    return {"treatment":"not_available_in_payload","name":"no_strict_strategy_contract","slots":[]}


def contract_risk(req):
    rid=req["requirement_id"]; t=txt(req)
    if rid=="IFRS_S2_26_C01":
        return {"treatment":"handled_by_report_design","name":"avoid_risk_management_duplication","slots":[]}
    if has(t,"inputs and parameters","data sources and the scope of operations"):
        return {"name":"risk_inputs_parameters","slots":[
            slot("data_sources",["physical_risk_exposures","metadata"],["data_source","scenario_basis","risk_rating_methodology"],description="Risk data sources and parameters",min_paths=2,max_paths=6),
            slot("scope",["physical_risk_exposures","value_chain_map"],["country","nace_code","node_type","upstream_downstream"],description="Scope of operations and exposures",min_paths=3,max_paths=8)]}
    if has(t,"scenario analysis"):
        return {"name":"risk_scenario_analysis","slots":[
            slot("scenario_links",["climate_risk_register","physical_risk_exposures"],["scenario_analysis_link","scenario_basis"],description="Scenario analysis linkage",min_paths=2,max_paths=6),
            slot("scenario_parameters",["physical_risk_exposures"],["assessment_horizon","acute_risk_score","chronic_risk_score"],description="Scenario risk parameters",min_paths=2,max_paths=6)]}
    if has(t,"nature, likelihood and magnitude","qualitative factors","quantitative thresholds"):
        return {"name":"risk_assessment_method","slots":[
            slot("likelihood_magnitude",["climate_risk_register"],["likelihood_score","severity_score","financial_impact_meur","risk_rating"],description="Likelihood and magnitude assessment",min_paths=4,max_paths=8),
            slot("methodology",["metadata"],["risk_rating_methodology"],description="Risk-rating thresholds and methodology")]}
    if has(t,"prioritises sustainability-related risks relative to other types of risk","prioritise climate-related risks relative"):
        return {"name":"risk_prioritisation","slots":[
            slot("risk_ratings",["climate_risk_register","metadata"],["risk_rating","risk_rating_methodology"],description="Risk prioritisation criteria",min_paths=2,max_paths=6),
            slot("relative_prioritisation",["governance","climate_risk_register"],["relative_to_other_risks","risk_appetite","priority_rank"],description="How climate risk is prioritised relative to other risk types")]}
    if has(t,"how the entity monitors","monitor climate-related risks"):
        return {"name":"risk_monitoring","slots":[slot("monitoring",["climate_risk_register"],["monitoring_frequency","risk_name","risk_rating"],description="Risk monitoring frequency and subject",min_paths=4,max_paths=8)]}
    if has(t,"changed the processes","compared with the previous reporting period"):
        return {"name":"risk_process_changes","slots":[slot("changes",["climate_risk_register"],["changed_since_prior_period"],description="Changes since prior period",min_paths=2,max_paths=6)]}
    if has(t,"processes the entity uses to identify, assess, prioritise and monitor sustainability-related opportunities",
           "processes to identify, assess, prioritise and monitor climate-related opportunities"):
        return {"name":"opportunity_process","slots":[slot("opportunity_process",["climate_opportunities"],["opportunity_type","monitoring_frequency","assessment"],description="Opportunity identification and monitoring process")]}
    if has(t,"integrated into and inform the entity's overall risk management","overall risk profile"):
        return {"name":"erm_integration","slots":[
            slot("record_level_integration",["climate_risk_register"],["erm_integrated_flag"],description="Extent of record-level ERM integration",min_paths=4,max_paths=8),
            slot("governance_integration",["governance"],["erm_integration_flag","major_transactions_climate_check"],description="Governance-level ERM integration",min_paths=1,max_paths=3)]}
    if has(t,"identify, assess, prioritise and monitor climate-related risks","identify, assess, prioritise and monitor sustainability-related risks"):
        return {"name":"risk_process_umbrella","slots":[
            slot("risk_identification",["climate_risk_register"],["risk_name","risk_description","risk_category"],description="Risk identification",min_paths=3,max_paths=6),
            slot("risk_assessment",["climate_risk_register"],["likelihood_score","severity_score","risk_rating","financial_impact"],description="Risk assessment",min_paths=3,max_paths=6),
            slot("risk_monitoring",["climate_risk_register"],["monitoring_frequency"],description="Risk monitoring",min_paths=2,max_paths=5)]}
    return {"treatment":"not_available_in_payload","name":"no_strict_risk_contract","slots":[]}


def _metrics_contract(name):
    C={
        "metrics_umbrella":{"name":"metrics_umbrella","slots":[
            slot("core_metrics",["reporting_kpis","scope1","scope2","scope3_travel","financed_emissions"],["scope1","scope2","scope3","financed_emissions","carbon_intensity","green_loans"],description="Core monitored metrics",min_paths=5,max_paths=10),
            slot("targets",["targets"],["target_type","metric","target_year","status"],description="Targets monitored using metrics",min_paths=2,max_paths=6)]},
        "metric_sources":{"name":"metric_sources","slots":[slot("sources",["ghg_methodology","metadata","targets"],["standard_reference","target_framework","pcaf_methodology"],description="Metric sources and frameworks",min_paths=2,max_paths=6)]},
        "industry_metrics":{"name":"industry_metrics","slots":[slot("banking_metrics",["reporting_kpis","financial_summary","financed_emissions"],["green_loans","high_carbon_sector_exposure","fossil_fuel_exposure","financed_emissions","carbon_intensity"],description="Industry-relevant banking metrics",min_paths=3,max_paths=8)]},
        "metric_definition":{"name":"metric_definition","slots":[slot("definition",["ghg_methodology","targets"],["measurement_approach","metric","scope","progress_metric"],description="Metric definition",min_paths=2,max_paths=6)]},
        "metric_type":{"name":"metric_measure_type","slots":[slot("measure_type",["targets","ghg_methodology"],["target_type","metric","scope"],value_terms=["absolute","intensity"],description="Absolute, relative or qualitative measure type",min_paths=2,max_paths=6)]},
        "metric_validation":{"name":"metric_validation","slots":[slot("validation",["targets","metadata"],["third_party_validated","validation_body","external_assurance"],description="Validation or assurance evidence",min_paths=2,max_paths=6)]},
        "metric_method":{"name":"metric_method_inputs_limitations","slots":[
            slot("method",["ghg_methodology","metadata"],["measurement_approach","key_inputs","key_assumptions","reason_for_approach","data_gaps"],description="Method, inputs, assumptions and limitations",min_paths=3,max_paths=8)]},
        "target_metric":{"name":"target_metric","slots":[slot("metric",["targets"],["metric","progress_metric"],description="Metric used to set and monitor target",min_paths=2,max_paths=5)]},
        "target_value":{"name":"target_value","slots":[slot("target_value",["targets"],["target_type","target_value_pct_reduction","target_year"],description="Specific target value",min_paths=3,max_paths=6)]},
        "target_period":{"name":"target_period","slots":[slot("period",["targets"],["baseline_year","target_year"],description="Target period",min_paths=2,max_paths=5)]},
        "target_base":{"name":"target_base_period","slots":[slot("base",["targets"],["baseline_year","baseline_value"],description="Target base period and value",min_paths=2,max_paths=5)]},
        "target_milestones":{"name":"target_milestones","slots":[slot("milestones",["targets"],["milestones_parsed","interim_milestones_json"],description="Interim milestones",min_paths=1,max_paths=6)]},
        "target_performance":{"name":"target_performance","coverage_cap":"partially_covered","slots":[
            slot("reported_progress",["targets","reporting_kpis"],["actual_progress","target_progress","progress_basis","status","schedule_elapsed"],description="Performance against target",min_paths=2,max_paths=8),
            slot("actual_verified_progress",["targets"],["actual_progress_pct_2024","target_progress_pct_2024"],description="Actual verified target progress",min_paths=1,max_paths=4)]},
        "metric_consistency":{"name":"metric_consistency","comparative":True,"slots":[
            slot("methods_over_time",["ghg_methodology"],["measurement_approach","changes_in_period","standard_reference"],description="Consistent methodology and changes",min_paths=2,max_paths=6),
            slot("comparatives",["scope1","scope2","financed_emissions"],["reporting_year","scope1_total","scope2_location","financed_em"],description="Comparative metric series",min_paths=4,max_paths=8,allow_generic=True)]},
        "comparative":{"name":"comparative_metrics","comparative":True,"slots":[slot("comparatives",["financial_summary","scope1","scope2","financed_emissions","internal_carbon_price"],["reporting_year","scope1_total","scope2_location","scope2_market","financed_em","carbon_price"],description="Prior-period comparative metrics",min_paths=5,max_paths=10,allow_generic=True)]},
        "comparative_limit":{"name":"comparative_data_limitations","slots":[slot("data_gaps",["metadata","reporting_kpis"],["data_gaps","comparative_available","scope1_fleet_included"],description="Comparative-data limitations",min_paths=1,max_paths=6)]},
        "scope1":{"name":"scope1_emissions","comparative":True,"slots":[
            slot("scope1_values",["scope1","reporting_kpis"],["scope1_total","scope1_2024","scope1_gas","scope1_fleet"],description="Scope 1 emissions",min_paths=2,max_paths=6)]},
        "scope2":{"name":"scope2_emissions","comparative":True,"slots":[
            slot("scope2_values",["scope2","reporting_kpis"],["scope2_location","scope2_market"],description="Scope 2 emissions",min_paths=3,max_paths=7)]},
        "scope3":{"name":"scope3_emissions","comparative":True,"slots":[
            slot("scope3_values",["scope3_categories","scope3_travel","reporting_kpis"],["emissions_tco2e","scope3_travel"],description="Scope 3 emissions",min_paths=3,max_paths=8),
            slot("categories",["scope3_categories"],["category_number","category_name","included_flag"],description="Scope 3 categories",min_paths=3,max_paths=8)]},
        "ghg_protocol":{"name":"ghg_protocol_and_gwp","slots":[slot("standard",["ghg_methodology"],["standard_reference","gwp_basis"],value_terms=["ghg protocol","ipcc"],description="GHG Protocol and GWP basis",min_paths=2,max_paths=6)]},
        "ghg_method":{"name":"ghg_measurement_method","slots":[
            slot("approach",["ghg_methodology"],["measurement_approach","key_inputs","key_assumptions","reason_for_approach"],description="GHG measurement approach, inputs and assumptions",min_paths=4,max_paths=8)]},
        "scope12_consolidation":{"name":"scope12_consolidation","slots":[
            slot("consolidated_group",["scope12_consolidation"],["consolidated_group_share_pct","consolidation_basis"],description="Consolidated accounting group share",min_paths=2,max_paths=6),
            slot("other_investees",["scope12_consolidation"],["other_investees_share_pct","note"],description="Other investees share",min_paths=2,max_paths=6)]},
        "scope2_contracts":{"name":"scope2_location_and_contractual_instruments","slots":[
            slot("location_based",["scope2","reporting_kpis"],["scope2_location"],description="Location-based Scope 2 emissions",min_paths=2,max_paths=5),
            slot("contractual_instruments",["metadata","ghg_methodology"],["scope2_rec_reconciliation","measurement_approach","key_inputs"],value_terms=["market-based","renewable energy certificates","ppa"],description="Contractual instruments and method",min_paths=1,max_paths=5)]},
        "scope3_categories":{"name":"scope3_categories","slots":[
            slot("categories",["scope3_categories"],["category_number","category_name","included_flag","emissions_tco2e"],description="Scope 3 categories and amounts",min_paths=5,max_paths=10)]},
        "scope3_method":{"name":"scope3_measurement_framework","slots":[
            slot("method",["scope3_categories","ghg_methodology"],["calculation_method","data_source","measurement_approach","key_inputs","key_assumptions"],description="Scope 3 measurement approach",min_paths=4,max_paths=10),
            slot("quality",["scope3_categories","financed_emissions_equity","financed_emissions_sovereign"],["pcaf_data_quality_score","data_source","proxy_confidence"],description="Scope 3 data quality",min_paths=2,max_paths=8)]},
        "financed":{"name":"financed_emissions","comparative":True,"slots":[
            slot("loan_financed_emissions",["financed_emissions","reporting_kpis"],["financed_em_loans","financed_emissions_2024","carbon_intensity"],description="Loan financed emissions",min_paths=3,max_paths=7),
            slot("asset_class_emissions",["financed_emissions_equity","financed_emissions_sovereign"],["asset_class","attributed_emissions","attribution_factor","pcaf_data_quality_score"],description="Financed emissions by asset class",min_paths=4,max_paths=10),
            slot("methodology",["metadata","financed_emissions_equity","financed_emissions_sovereign"],["pcaf_methodology","proxy_basis","proxy_reason","data_gap_flag"],description="PCAF methodology and limitations",min_paths=2,max_paths=8)]},
        "category15":{"name":"scope3_category15_financed_emissions","slots":[
            slot("category15",["scope3_categories"],["category_number","category_name","emissions_tco2e"],value_terms=["investments","category 15"],description="Category 15 emissions",min_paths=2,max_paths=6),
            slot("financed_subtotal",["financed_emissions","reporting_kpis"],["financed_em"],description="Financed-emissions subtotal",min_paths=2,max_paths=5)]},
        "transition_metric":{"name":"transition_risk_exposure_metric","slots":[slot("transition_exposure",["reporting_kpis"],["high_carbon_sector_exposure","fossil_fuel_exposure"],description="Amount and percentage exposed to transition risk",min_paths=2,max_paths=6)]},
        "physical_metric":{"name":"physical_risk_exposure_metric","slots":[slot("physical_exposure",["reporting_kpis"],["physical_risk","physical_exposure"],description="Amount and percentage vulnerable to physical risk")]},
        "opportunity_metric":{"name":"climate_opportunity_metric","slots":[slot("opportunity_metrics",["financial_summary","reporting_kpis"],["green_loans","climate_capex"],description="Assets or activities aligned with climate opportunities",min_paths=2,max_paths=6)]},
        "capital":{"name":"climate_capital_deployment","comparative":True,"slots":[slot("capital",["financial_summary","reporting_kpis"],["climate_capex","climate_opex","green_loans"],description="Climate-related capital deployment and financing",min_paths=3,max_paths=8)]},
        "carbon_price":{"name":"internal_carbon_price","comparative":True,"slots":[
            slot("price",["internal_carbon_price"],["carbon_price_eur_per_tco2e","currency"],description="Internal carbon price",min_paths=2,max_paths=5),
            slot("application",["internal_carbon_price"],["price_type","application_scope","applies_to_lending_decisions","applies_to_financed_emissions","benchmark_reference","review_frequency"],description="How internal carbon price is applied",min_paths=3,max_paths=8)]},
        "remuneration_how":{"name":"remuneration_considerations","coverage_cap":"partially_covered","slots":[
            slot("percentage",["reporting_kpis"],["ceo_esg_compensation_pct"],description="Climate-linked remuneration percentage"),
            slot("how_factored",["reporting_kpis"],["remuneration_policy","climate_remuneration_method"],description="How climate considerations are factored into remuneration")]},
        "remuneration_pct":{"name":"remuneration_percentage","slots":[slot("percentage",["reporting_kpis"],["ceo_esg_compensation_pct"],description="Percentage of executive remuneration linked to climate") ]},
        "commercial":{"name":"commercial_banking_financed_emissions","coverage_cap":"partially_covered","slots":[
            slot("financed_emissions",["financed_emissions","reporting_kpis"],["financed_em","carbon_intensity"],description="Commercial-banking financed emissions",min_paths=2,max_paths=6),
            slot("industry_asset_disaggregation",["financed_emissions_equity","financed_emissions_sovereign"],["asset_class","nace_code","attributed_emissions"],description="Industry and asset-class disaggregation",min_paths=3,max_paths=8),
            slot("loan_level_disaggregation",["commercial_banking_exposures"],["industry","asset_class","scope1","scope2","scope3"],description="Loan-book financed emissions disaggregated by industry and asset class")]},
        "commercial_exposure":{"name":"commercial_banking_gross_exposure","coverage_cap":"partially_covered","slots":[
            slot("exposures",["financial_summary","financed_emissions_equity","financed_emissions_sovereign"],["total_loans","total_undrawn","market_value","nominal_amount","asset_class","nace_code"],description="Gross exposure by asset class or industry",min_paths=3,max_paths=8),
            slot("full_industry_asset_breakdown",["commercial_banking_exposures"],["industry","asset_class","gross_exposure"],description="Complete commercial-banking exposure breakdown")]},
        "commercial_coverage":{"name":"commercial_banking_calculation_coverage","slots":[slot("coverage_percentage",["reporting_kpis","financed_emissions"],["gross_exposure_included_pct","financed_emissions_coverage_pct"],description="Percentage of gross exposure included in calculation")]},
        "commercial_method":{"name":"commercial_banking_methodology","slots":[slot("methodology",["metadata"],["pcaf_methodology"],description="Financed-emissions allocation methodology",min_paths=1,max_paths=5)]},
        "classification":{"name":"industry_classification","coverage_cap":"partially_covered","slots":[
            slot("codes",["financed_emissions_equity","financed_emissions_sovereign"],["nace_code"],description="Industry classification codes",min_paths=2,max_paths=6),
            slot("system_name",["metadata"],["industry_classification_system"],description="Named industry-classification system")]},
        "asset_classes":{"name":"included_asset_classes","coverage_cap":"partially_covered","slots":[
            slot("observed_classes",["financed_emissions_equity","financed_emissions_sovereign","financial_summary"],["asset_class","total_loans","total_undrawn"],description="Observed asset classes",min_paths=3,max_paths=8),
            slot("complete_classes",["commercial_banking_exposures"],["project_finance","bonds","equity","undrawn"],description="Complete required asset-class scope")]},
        "asset_management":{"name":"asset_management_financed_emissions","coverage_cap":"partially_covered","slots":[
            slot("investment_emissions",["financed_emissions_equity","financed_emissions_sovereign"],["attributed_emissions","asset_class","market_value"],description="Investment financed emissions",min_paths=3,max_paths=8),
            slot("aum",["reporting_kpis"],["assets_under_management","aum"],description="Assets under management denominator")]},
        "cross_section_control":{"treatment":"handled_by_report_design","name":"cross_section_metric_preparation_control","slots":[]},
        "target_identity":{"name":"climate_target_identity","slots":[slot("target",["targets"],["target_type","metric","target_value_pct_reduction","target_year"],description="Target definition and value",min_paths=3,max_paths=8)]},
        "target_objective":{"name":"target_objective","slots":[slot("objective",["targets"],["target_type","target_framework"],description="Target objective and framework",min_paths=2,max_paths=5)]},
        "target_scope":{"name":"target_scope","slots":[slot("scope",["targets"],["target_applies_to","scope","scope_coverage"],description="Target application scope",min_paths=2,max_paths=6)]},
        "target_abs_intensity":{"name":"target_absolute_or_intensity","slots":[slot("type",["targets"],["target_type"],value_terms=["absolute","intensity"],description="Absolute or intensity target type")]},
        "target_paris":{"name":"target_paris_alignment","slots":[slot("alignment",["targets"],["latest_paris_alignment","target_framework"],description="Paris alignment and framework",min_paths=2,max_paths=5)]},
        "target_validation":{"name":"target_validation","slots":[slot("validation",["targets"],["third_party_validated","validation_body"],description="Third-party target validation",min_paths=2,max_paths=5)]},
        "target_review":{"name":"target_review_process","slots":[slot("review",["targets"],["review_frequency"],description="Target review process")]},
        "target_progress_metric":{"name":"target_progress_metric","slots":[slot("progress_metric",["targets"],["progress_metric","metric"],description="Metric used to monitor target progress",min_paths=2,max_paths=5)]},
        "target_ghg_gases":{"name":"target_ghg_gases","slots":[slot("gases",["targets"],["ghg_gases_covered"],description="Greenhouse gases covered by target")]},
        "target_ghg_scopes":{"name":"target_ghg_scopes","slots":[slot("scopes",["targets"],["scope_coverage","scope"],description="Scopes covered by target",min_paths=2,max_paths=5)]},
        "target_gross_net":{"name":"target_gross_net","slots":[slot("gross_net",["targets"],["gross_or_net","associated_gross_target_id"],description="Gross/net target basis",min_paths=1,max_paths=5)]},
        "target_sectoral":{"name":"target_sectoral_decarbonisation","slots":[slot("sectoral",["targets"],["sectoral_decarbonisation"],description="Sectoral decarbonisation approach")]},
        "target_credits":{"name":"target_planned_carbon_credits","slots":[
            slot("planned_use",["targets"],["planned_carbon_credits_pct","planned_credit_type"],description="Planned carbon-credit use",min_paths=2,max_paths=5),
            slot("credit_characteristics",["carbon_credits"],["credit_type","underlying_mechanism","permanence_rating","additionality_verified","registry"],description="Carbon-credit characteristics",min_paths=3,max_paths=8)]},
        "carbon_credits":{"name":"carbon_credits","slots":[
            slot("credit_quantity_use",["carbon_credits","reporting_kpis"],["tonnes_co2e","use","retired_tco2e","total_credits"],description="Carbon-credit quantity and use",min_paths=3,max_paths=8),
            slot("credit_quality",["carbon_credits","reporting_kpis"],["registry","project_type","credit_type","underlying_mechanism","permanence","additionality"],description="Carbon-credit quality characteristics",min_paths=4,max_paths=10)]},
    }
    return C[name]


mb = {}

def contract_metrics(req):
    rid=req["requirement_id"]; t=txt(req)
    # IFRS S1 direct routing
    exact={
        "IFRS_S1_46_C01":"metrics_umbrella","IFRS_S1_46_C03":"metrics_umbrella","IFRS_S1_46_C04":"metrics_umbrella",
        "IFRS_S1_47_C01":"metric_sources","IFRS_S1_48_C01":"industry_metrics","IFRS_S1_49_C01":"metric_sources",
        "IFRS_S1_50_C01":"metric_definition","IFRS_S1_50_C02":"metric_type","IFRS_S1_50_C03":"metric_validation","IFRS_S1_50_C04":"metric_method",
        "IFRS_S1_51_C01":"target_metric","IFRS_S1_51_C02":"target_value","IFRS_S1_51_C03":"target_period","IFRS_S1_51_C04":"target_base",
        "IFRS_S1_51_C05":"target_milestones","IFRS_S1_51_C06":"target_performance",
        "IFRS_S1_52_C01":"metric_consistency","IFRS_S1_B49_C01":"comparative","IFRS_S1_B54_C01":"comparative_limit",
        "IFRS_S2_28_C01":"metrics_umbrella","IFRS_S2_28_C02":"industry_metrics","IFRS_S2_28_C03":"target_identity",
        "IFRS_S2_29_C03":"scope1","IFRS_S2_29_C04":"scope2","IFRS_S2_29_C05":"scope3",
        "IFRS_S2_29_C07":"ghg_protocol","IFRS_S2_29_C09":"ghg_method","IFRS_S2_29_C10":"ghg_method",
        "IFRS_S2_29_C13":"scope12_consolidation","IFRS_S2_29_C14":"scope12_consolidation","IFRS_S2_29_C15":"scope12_consolidation",
        "IFRS_S2_29_C16":"scope2_contracts","IFRS_S2_29_C18":"scope3_categories","IFRS_S2_29_C19":"scope3_method",
        "IFRS_S2_29_C20":"transition_metric","IFRS_S2_29_C21":"physical_metric","IFRS_S2_29_C22":"opportunity_metric",
        "IFRS_S2_29_C23":"capital","IFRS_S2_29_C25":"carbon_price","IFRS_S2_29_C26":"carbon_price",
        "IFRS_S2_29_C28":"remuneration_how","IFRS_S2_29_C29":"remuneration_pct","IFRS_S2_29C_C01":"category15",
        "IFRS_S2_30_C01":"metric_method","IFRS_S2_31_C01":"cross_section_control","IFRS_S2_32_C01":"industry_metrics",
        "IFRS_S2_33_C01":"target_metric","IFRS_S2_33_C02":"target_objective","IFRS_S2_33_C03":"target_scope",
        "IFRS_S2_33_C04":"target_period","IFRS_S2_33_C05":"target_base","IFRS_S2_33_C06":"target_milestones",
        "IFRS_S2_33_C07":"target_abs_intensity","IFRS_S2_33_C08":"target_paris",
        "IFRS_S2_34_C01":"target_validation","IFRS_S2_34_C02":"target_review","IFRS_S2_34_C03":"target_progress_metric",
        "IFRS_S2_35_C01":"target_performance","IFRS_S2_36_C01":"target_ghg_gases","IFRS_S2_36_C02":"target_ghg_scopes",
        "IFRS_S2_36_C03":"target_gross_net","IFRS_S2_36_C04":"target_sectoral",
        "IFRS_S2_36_C06":"target_credits","IFRS_S2_36_C07":"target_credits","IFRS_S2_36_C08":"target_credits","IFRS_S2_36_C09":"target_credits",
        "IFRS_S2_37_C01":"industry_metrics",
        "IFRS_S2_B20_C01":"metrics_umbrella","IFRS_S2_B21_C01":"ghg_protocol","IFRS_S2_B23_C02":"ghg_protocol","IFRS_S2_B23_C03":"ghg_protocol",
        "IFRS_S2_B26_C01":"ghg_method","IFRS_S2_B27_C01":"ghg_method","IFRS_S2_B28_C01":"ghg_method","IFRS_S2_B28_C02":"ghg_method",
        "IFRS_S2_B29_C01":"ghg_method","IFRS_S2_B30_C01":"scope2_contracts","IFRS_S2_B32_C01":"scope3_categories","IFRS_S2_B33_C01":"scope3_categories",
        "IFRS_S2_B36_C01":"scope3_method","IFRS_S2_B37_C01":"financed","IFRS_S2_B38_C01":"scope3_method","IFRS_S2_B39_C01":"scope3_method",
        "IFRS_S2_B40_C01":"scope3_method","IFRS_S2_B40_C02":"scope3_method","IFRS_S2_B40_C03":"scope3_method","IFRS_S2_B40_C04":"scope3_method",
        "IFRS_S2_B43_C01":"scope3_method","IFRS_S2_B47_C01":"scope3_method","IFRS_S2_B49_C01":"scope3_method","IFRS_S2_B53_C01":"scope3_method",
        "IFRS_S2_B55_C01":"scope3_method","IFRS_S2_B56_C01":"scope3_method","IFRS_S2_B56_C02":"scope3_method",
        "IFRS_S2_B59_C01":"financed","IFRS_S2_B60_C01":"financed",
        "IFRS_S2_B61_C01":"asset_management","IFRS_S2_B61_C02":"asset_management","IFRS_S2_B61_C03":"asset_management","IFRS_S2_B61_C04":"asset_management",
        "IFRS_S2_B62_C01":"commercial","IFRS_S2_B62_C03":"commercial_exposure","IFRS_S2_B62_C04":"commercial_exposure",
        "IFRS_S2_B62_C06":"commercial_coverage","IFRS_S2_B62_C07":"commercial_coverage","IFRS_S2_B62_C08":"commercial_coverage",
        "IFRS_S2_B62_C09":"commercial_method","IFRS_S2_B62A_C05":"classification","IFRS_S2_B62A_C06":"classification","IFRS_S2_B62A_C07":"asset_classes",
        "IFRS_S2_B64_C01":"metrics_umbrella",
        "IFRS_S2_B65_C01":"cross_section_control","IFRS_S2_B65_C02":"cross_section_control","IFRS_S2_B65_C03":"cross_section_control",
        "IFRS_S2_B65_C04":"cross_section_control","IFRS_S2_B65_C05":"cross_section_control","IFRS_S2_B65_C06":"cross_section_control",
        "IFRS_S2_B65_C08":"cross_section_control","IFRS_S2_B65_C09":"cross_section_control",
        "IFRS_S2_B66_C01":"target_identity","IFRS_S2_B67_C01":"target_metric","IFRS_S2_B68_C01":"target_gross_net",
        "IFRS_S2_B69_C01":"target_gross_net","IFRS_S2_B70_C01":"target_credits","IFRS_S2_B71_C01":"carbon_credits",
    }
    conditional_ids={
        "IFRS_S1_51_C07","IFRS_S1_B50_C01","IFRS_S1_B50_C02","IFRS_S1_B50_C03",
        "IFRS_S1_B52_C01","IFRS_S1_B52_C02","IFRS_S1_B52_C03","IFRS_S1_B53_C01",
        "IFRS_S2_29_C11","IFRS_S2_29B_C01","IFRS_S2_29B_C02","IFRS_S2_34_C04",
        "IFRS_S2_B34_C01","IFRS_S2_B34_C02","IFRS_S2_B34_C03","IFRS_S2_B57_C01",
    }
    insurance_ids={rid for rid in mb if rid.startswith("IFRS_S2_B63")}
    if rid in conditional_ids:
        return {"treatment":"conditional_not_triggered","name":"conditional_metric_or_target_change","slots":[]}
    if rid=="IFRS_S1_53_C01":
        return {"treatment":"handled_by_report_design","name":"metric_labeling_control","slots":[]}
    if rid in insurance_ids:
        return {"treatment":"not_applicable_to_entity_scope","name":"insurance_metrics_not_in_payload_scope","slots":[]}
    if rid in exact:
        return _metrics_contract(exact[rid])
    return {"treatment":"not_available_in_payload","name":"no_strict_metrics_contract","slots":[]}


_old_contract_metrics = contract_metrics

def contract_metrics(req):
    rid=req["requirement_id"]
    if rid.startswith("IFRS_S2_B63"):
        return {"treatment":"not_applicable_to_entity_scope","name":"insurance_metrics_not_in_payload_scope","slots":[]}
    return _old_contract_metrics(req)


CONTRACTORS = {
    "general_requirements": contract_general,
    "governance": contract_governance,
    "strategy": contract_strategy,
    "risk_management": contract_risk,
    "metrics_and_targets": contract_metrics,
}

In [ ]:
def map_section(section,payload,req_doc):
    flat=flatten_records(payload)
    year=payload.get("metadata",{}).get("reporting_year")
    mappings=[]
    for req in iter_requirements(req_doc):
        c=CONTRACTORS[section](req)
        treatment=c.get("treatment","data_mapping")
        row={
            "requirement_id":req["requirement_id"],
            "standard":req.get("standard"),
            "paragraph_id":req.get("paragraph_id"),
            "requirement_text":req.get("requirement_text"),
            "mapping_contract":c.get("name"),
            "mapping_treatment":treatment,
            "mapping_status":None,
            "coverage_cap":c.get("coverage_cap"),
            "coverage_note":"",
            "evidence_paths":[],
            "evidence_values":[],
            "required_evidence_slots":[],
            "satisfied_evidence_slots":[],
            "missing_evidence_slots":[],
        }
        if treatment!="data_mapping":
            row["mapping_status"]=treatment
            if treatment=="handled_by_report_design":
                row["coverage_note"]="Validated during report planning/rendering rather than by mapping an entity payload field."
            elif treatment=="conditional_not_triggered":
                row["coverage_note"]="The requirement is conditional and the triggering event is not represented in the current payload."
            elif treatment=="not_applicable_to_entity_scope":
                row["coverage_note"]="The payload does not contain the business activity to which this requirement applies."
            mappings.append(row)
            continue
        comparative=c.get("comparative",False) or has(txt(req),"comparative","preceding period","prior period")
        all_paths=[]
        for sl in c.get("slots",[]):
            paths=choose_paths(flat,sl,year,comparative=comparative)
            sat=len(paths)>=sl["min_paths"]
            row["required_evidence_slots"].append({
                "slot":sl["name"],
                "description":sl["description"],
                "minimum_paths":sl["min_paths"],
            })
            (row["satisfied_evidence_slots"] if sat else row["missing_evidence_slots"]).append(sl["name"])
            all_paths.extend(paths)
        seen=set()
        all_paths=[x for x in all_paths if not (x in seen or seen.add(x))]
        idx={e["path"]:e["value"] for e in flat}
        row["evidence_paths"]=all_paths
        row["evidence_values"]=[{"payload_path":p,"value":idx[p]} for p in all_paths]
        total=len(c.get("slots",[]))
        sat=len(row["satisfied_evidence_slots"])
        if total and sat==total:
            status="covered"
        elif sat>0:
            status="partially_covered"
        else:
            status="not_available_in_payload"
        if c.get("coverage_cap")=="partially_covered" and status=="covered":
            status="partially_covered"
            row["coverage_note"]="Evidence paths exist, but strict semantic policy caps this requirement at partial because an explicit process/policy or complete disaggregation is absent."
        elif status=="partially_covered":
            row["coverage_note"]="At least one required evidence slot is supported, but one or more required slots are missing."
        elif status=="not_available_in_payload":
            row["coverage_note"]="No required evidence slot is supported by the current section payload."
        else:
            row["coverage_note"]="All strict evidence slots are supported by resolvable, non-empty payload paths."
        row["mapping_status"]=status
        mappings.append(row)
    return mappings


In [ ]:

payloads = {
    section: load_json(files["payload"])
    for section, files in RESOLVED_FILES.items()
}
requirements_docs = {
    section: load_json(files["requirements"])
    for section, files in RESOLVED_FILES.items()
}

mappings_by_section = {
    section: map_section(section, payloads[section], requirements_docs[section])
    for section in RESOLVED_FILES
}

summary = {
    "total_requirements": sum(len(rows) for rows in mappings_by_section.values()),
    "status_totals": dict(Counter(
        row["mapping_status"]
        for rows in mappings_by_section.values()
        for row in rows
    )),
    "sections": {},
}

for section, rows in mappings_by_section.items():
    counts = Counter(row["mapping_status"] for row in rows)
    summary["sections"][section] = {
        "requirement_count": len(rows),
        "status_counts": dict(counts),
        "not_available_requirement_ids": [
            row["requirement_id"]
            for row in rows
            if row["mapping_status"] == "not_available_in_payload"
        ],
        "partially_covered_requirement_ids": [
            row["requirement_id"]
            for row in rows
            if row["mapping_status"] == "partially_covered"
        ],
    }

print(json.dumps(summary, indent=2))


In [ ]:

def build_tests(section, payload, requirements_doc, mappings):
    source_requirements = iter_requirements(requirements_doc)
    source_ids = [row["requirement_id"] for row in source_requirements]
    mapped_ids = [row["requirement_id"] for row in mappings]
    flat_index = {entry["path"]: entry["value"] for entry in flatten_records(payload)}

    invalid_paths = []
    empty_paths = []
    value_mismatches = []

    for row in mappings:
        for item in row["evidence_values"]:
            path = item["payload_path"]
            if path not in flat_index:
                invalid_paths.append({
                    "requirement_id": row["requirement_id"],
                    "payload_path": path,
                })
                continue
            actual = flat_index[path]
            if is_empty(actual):
                empty_paths.append({
                    "requirement_id": row["requirement_id"],
                    "payload_path": path,
                })
            if actual != item["value"]:
                value_mismatches.append({
                    "requirement_id": row["requirement_id"],
                    "payload_path": path,
                    "stored_value": item["value"],
                    "actual_value": actual,
                })

    tests = {
        "requirement_count_matches": (
            len(mappings)
            == requirements_doc["row_count"]
            == len(source_requirements)
        ),
        "all_requirement_ids_mapped": set(mapped_ids) == set(source_ids),
        "requirement_ids_unique": len(mapped_ids) == len(set(mapped_ids)),
        "all_evidence_paths_resolve": not invalid_paths,
        "all_selected_evidence_non_empty": not empty_paths,
        "stored_evidence_values_match_payload": not value_mismatches,
        "covered_rows_have_satisfied_slots": all(
            row["satisfied_evidence_slots"]
            for row in mappings
            if row["mapping_status"] == "covered"
        ),
        "non_data_controls_have_no_evidence_paths": all(
            not row["evidence_paths"]
            for row in mappings
            if row["mapping_treatment"] != "data_mapping"
        ),
    }

    return {
        "section": section,
        "result": "PASS" if all(tests.values()) else "FAIL",
        "tests": tests,
        "invalid_paths": invalid_paths,
        "empty_paths": empty_paths,
        "value_mismatches": value_mismatches,
    }

test_reports = {
    section: build_tests(
        section,
        payloads[section],
        requirements_docs[section],
        mappings_by_section[section],
    )
    for section in mappings_by_section
}

combined_test_report = {
    "result": (
        "PASS"
        if all(report["result"] == "PASS" for report in test_reports.values())
        else "FAIL"
    ),
    "sections": test_reports,
}

OUTPUT_DIR = MAPPING_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for section, rows in mappings_by_section.items():
    (OUTPUT_DIR / f"{section}_requirement_mapping_v1.json").write_text(
        json.dumps(rows, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    (OUTPUT_DIR / f"{section}_mapping_test_report_v1.json").write_text(
        json.dumps(test_reports[section], indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

(OUTPUT_DIR / "all_sections_requirement_mapping_v1.json").write_text(
    json.dumps(mappings_by_section, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(OUTPUT_DIR / "all_sections_mapping_summary_v1.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(OUTPUT_DIR / "all_sections_mapping_test_report_v1.json").write_text(
    json.dumps(combined_test_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("TEST RESULT:", combined_test_report["result"])
print("Saved outputs to:", OUTPUT_DIR.resolve())
